# Phase 2 — Baseline + advanced learners, first Qini curves

**Goal.** Train five uplift-related estimators on the Criteo dataset, evaluate
them with Qini curves on a shared held-out test set, and show that proper uplift
models beat the propensity-only strawman.

**The five models:**

1. **Propensity baseline** — rank by predicted P(convert), ignore treatment. The "wrong" strategy from the Blake/Nosko/Tadelis paper. We include it to prove it loses.
2. **S-learner** — one model with treatment as an extra feature. Uplift = μ(x,1) − μ(x,0).
3. **T-learner** — two models, one per arm. Uplift = μ₁(x) − μ₀(x).
4. **Class Transformation** (Jaskowski & Jaroszewicz 2012) — recast uplift as a single weighted binary classification.
5. **X-learner** (Künzel et al. 2019) — two-stage estimator with imputed counterfactuals; designed for imbalanced treatment allocation (which is what we have — 85/15).

The Causal Forest lands in Phase 3 (needs a downsample because of memory).

**Base learner.** All five use `HistGradientBoostingClassifier` /
`Regressor` from scikit-learn as the underlying gradient-booster.
This is the same algorithmic family as LightGBM (histogram-based GBDT) but ships
with OpenMP bundled in the sklearn wheel, so it works out-of-the-box on macOS
without needing `brew install libomp`.

**Target outcome:** `conversion` (~0.3% base rate). Rare, but the ATE is
statistically real (from Phase 1: +0.115 pp, 95% CI [+0.108, +0.122]).

In [ ]:
import sys
import time
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.uplift.data import load_criteo, FEATURE_COLS
from src.uplift.split import make_split
from src.uplift.evaluation import qini_curve, qini_coefficient
from src.uplift.plots import plot_qini_curves
from src.uplift.learners import (
    PropensityBaseline,
    SLearner,
    TLearner,
    ClassTransformationLearner,
    XLearner,
)

sns.set_theme(style="whitegrid", context="notebook", palette="deep")

## 1. Load and split the data

80/20 train/test split, stratified by treatment so both arms are represented
in the same proportion in each subset. Seed is pinned (42) — every learner
will see the same rows in train and the same rows in test, which is what
makes the Qini comparison honest.

In [ ]:
df = load_criteo()
train, test = make_split(df)

print(f"train: {len(train):>10,} rows  ({train['treatment'].mean():.1%} treated)")
print(f"test:  {len(test):>10,} rows  ({test['treatment'].mean():.1%} treated)")
print(f"conversion base rate — train: {train['conversion'].mean():.4%}, test: {test['conversion'].mean():.4%}")

## 2. Prepare X, T, Y arrays

All learners expect three arrays: feature matrix `X`, binary treatment `T`,
binary outcome `Y`. We use `conversion` as the outcome since it's the
business-relevant one (visits are a leading indicator, conversions are the money).

In [ ]:
X_tr = train[FEATURE_COLS].to_numpy()
T_tr = train["treatment"].to_numpy()
Y_tr = train["conversion"].to_numpy()

X_te = test[FEATURE_COLS].to_numpy()
T_te = test["treatment"].to_numpy()
Y_te = test["conversion"].to_numpy()

print(f"X_tr: {X_tr.shape}  X_te: {X_te.shape}")
print(f"Y_tr conversions: {Y_tr.sum():,}   Y_te conversions: {Y_te.sum():,}")

## 3. Fit all five models

Each learner exposes the same interface — `.fit(X, T, Y)` and `.predict_uplift(X)`.
This uniform API is what lets us iterate over them cleanly and compare Qini
curves apples-to-apples.

**Expected fit time (M2 laptop):**

| model | fits inside | rough time |
|---|---|---|
| Propensity baseline | 1 model on 11M rows | ~1 min |
| S-learner | 1 model on 11M rows | ~1 min |
| T-learner | 2 models (1.7M control, 9.5M treated) | ~1 min |
| Class Transformation | 1 weighted model on 11M | ~1 min |
| X-learner | 5 models (2 outcome, 2 pseudo-uplift, 1 propensity) | ~3 min |

Total ~5–8 min. Grab coffee.

In [ ]:
models = {
    "Propensity (baseline)": PropensityBaseline(),
    "S-learner":              SLearner(),
    "T-learner":              TLearner(),
    "Class Transformation":   ClassTransformationLearner(),
    "X-learner":              XLearner(),
}

fit_times = {}
for name, m in models.items():
    t0 = time.time()
    m.fit(X_tr, T_tr, Y_tr)
    fit_times[name] = time.time() - t0
    print(f"  {name:<25s} fit in {fit_times[name]:6.1f}s")

## 4. Predict uplift on the held-out test set

Each model produces a predicted uplift per test-set user. Then we compute the
Qini curve for each. Recall from the guide:

$$Q(k) = n_t(k) - n_c(k) \cdot \frac{N_t(k)}{N_c(k)}$$

where $n_t, n_c$ are cumulative conversions among treated/control in the
top-$k$ ranked users and $N_t, N_c$ are cumulative counts. This is
"incremental conversions attributed to targeting the top $k$ users".

In [ ]:
curves = {}
coefs = {}
for name, m in models.items():
    u = m.predict_uplift(X_te)
    curve = qini_curve(Y_te, T_te, u)
    curves[name] = curve
    coefs[name] = qini_coefficient(curve)

coef_df = pd.DataFrame({
    "model": list(coefs.keys()),
    "qini_coefficient": list(coefs.values()),
    "fit_time_s": [fit_times[n] for n in coefs.keys()],
}).sort_values("qini_coefficient", ascending=False).reset_index(drop=True)

coef_df.style.format({"qini_coefficient": "{:+.2f}", "fit_time_s": "{:.1f}"}) \
    .bar(subset=["qini_coefficient"], color="#b6532c", align=0)

## 5. The Qini curves — the money-shot plot

The Qini curve reads left-to-right as "if we only target the top X% by
predicted uplift, how many incremental conversions do we generate?"

- **Above the dashed random-targeting line** = model is picking real persuadables
- **On the line** = model is no better than random
- **Below the line** = model is anti-picking; targeting the worst first

The story we expect (from the Blake paper and Diemert et al.): uplift models
(X-learner, T-learner) should visibly beat the propensity baseline, which is
the standard "target likely converters" strategy that industry gets wrong.

In [ ]:
# Order curves so the highest Qini coefficient plots on top with the accent color
ordered = dict(sorted(curves.items(), key=lambda kv: -coefs[kv[0]]))

fig, ax = plt.subplots(figsize=(8.5, 5.5))
plot_qini_curves(ordered, ax=ax, title="Qini curves — Criteo conversion, held-out 2.8M test set")
plt.tight_layout()
plt.show()

## 6. The business question — top-K targeting

The Qini coefficient is a great single-number summary, but the actionable
question for a marketer is: *if I can only afford to target N% of users,
how many incremental conversions do I capture?*

We report, for each model, the incremental conversions captured at 10%, 20%,
and 50% targeting — and the fraction of the total-population lift that
represents.

In [ ]:
def top_k_lift(curve: pd.DataFrame, k_frac: float) -> float:
    """Cumulative incremental conversions when targeting top k_frac of users."""
    idx = (curve["share"] >= k_frac).idxmax()
    return float(curve["qini"].iloc[idx])

total_qini = curves["S-learner"]["qini"].iloc[-1]  # same for all models

rows = []
for name, curve in curves.items():
    row = {"model": name}
    for frac in [0.10, 0.20, 0.50]:
        row[f"lift@{int(frac*100)}%"] = top_k_lift(curve, frac)
        row[f"share@{int(frac*100)}%"] = top_k_lift(curve, frac) / total_qini
    rows.append(row)

topk_df = pd.DataFrame(rows).set_index("model")
topk_df = topk_df.reindex(coef_df["model"])  # order by qini coefficient

styled = topk_df.style.format({
    "lift@10%": "{:+.1f}", "lift@20%": "{:+.1f}", "lift@50%": "{:+.1f}",
    "share@10%": "{:.1%}", "share@20%": "{:.1%}", "share@50%": "{:.1%}",
})
styled

## 7. Interpretation and Phase 3 preview

**Phase 2 done when:**
- All five learners fit and predict without errors ✓
- Qini curves computed on the same held-out 2.8M test set ✓
- The Qini-coefficient table has an unambiguous ranking ✓
- The top-K analysis translates the Qini into a business number ✓

**What we're learning:**
- If the propensity baseline is at or near the top, the story is different
  from the Blake paper on this dataset — worth calling out honestly in the writeup.
- If X-learner beats T-learner cleanly, that confirms the imbalance-handling
  intuition (Criteo is 85/15 treated).
- The Class Transformation is a wild-card: when it works, it's elegant; when
  it doesn't, the weighting correction usually explains why.

**Phase 3:** train the Causal Forest on a ~500k downsample of the training
set (memory constraint on M2 laptops), evaluate on the *same* 2.8M test set,
and see whether it makes the ranking table again.

**Phase 4:** bootstrap confidence intervals on the Qini coefficient (so the
ranking is defensible under noise), calibration check on predicted uplifts,
and the final README with the headline figure.